# LeetCode 949 — Largest Time for Given Digits bằng Python

Link đề: https://leetcode.com/problems/largest-time-for-given-digits/ · Độ khó: **Medium** · Chủ đề: Array, String, Enumeration

Skeleton LeetCode:

```python
class Solution:
    def largestTimeFromDigits(self, arr: List[int]) -> str:
        ...
```

Ghi chú: đáp án duy nhất (một chuỗi `"HH:MM"` hoặc `""`). Điểm cốt lõi: chỉ có `4! = 24` hoán vị của `4` chữ số (hoặc `24 · 60 = 1440` giờ hợp lệ), nên **liệt kê** là đủ; ta chọn thời điểm **muộn nhất** (so theo số phút hoặc theo chuỗi `"HH:MM"`, hai cách tương đương). Cách thứ ba: **tham lam từng chữ số** có kiểm tra khả thi. Oracle: **duyệt mọi bộ 4 chữ số `abcd ∈ [0000, 9999]`** có cùng đa tập chữ số với `arr` và tạo thành giờ hợp lệ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: dùng **đúng một lần** mỗi chữ số trong `arr` (4 chữ số) để tạo thời gian `"HH:MM"` 24 giờ hợp lệ muộn nhất (`00:00 … 23:59`); không có thì trả `""`.
2. Nắm công cụ Python: `itertools.permutations`, `max` với khoá, định dạng `f"{h:02d}:{m:02d}"`, `Counter`, tham lam có kiểm tra khả thi.
3. Đi từ **thử 24 hoán vị** → **duyệt `1440` thời điểm từ muộn đến sớm và so đa tập chữ số** → **tham lam chọn chữ số lớn nhất ở từng vị trí (có kiểm tra phần còn lại)** → **quay lui chọn chữ số theo thứ tự giảm dần**.
4. Hiểu mấu chốt: "muộn nhất" là so sánh theo **tổng số phút** `60·H + M` (hoặc từ điển chuỗi `"HH:MM"` vì độ dài cố định); điều kiện hợp lệ `H ≤ 23`, `M ≤ 59`.
5. Tránh lỗi kinh điển: chấp nhận `H = 24..29` hoặc `M = 60..69`, dùng một chữ số hai lần (hoặc bỏ sót chữ số trùng), tham lam chọn chữ số đầu lớn nhất mà bỏ qua việc phần còn lại phải lập được phút hợp lệ, trả `None` thay vì `""`, và quên chữ số `0` đầu (`6:26` phải là `06:26`).

## 2. Đề bài & ví dụ

Cho mảng `arr` gồm đúng `4` chữ số. Tìm thời gian 24 giờ **muộn nhất** có thể tạo được, dùng **mỗi chữ số đúng một lần**.

Thời gian 24 giờ có dạng `"HH:MM"` với `HH` từ `00` đến `23` và `MM` từ `00` đến `59`. Thời điểm sớm nhất là `00:00`, muộn nhất là `23:59`.

Trả về thời gian muộn nhất ở dạng `"HH:MM"`; nếu không tạo được giờ hợp lệ nào, trả về chuỗi rỗng `""`.

**Ví dụ 1:** `arr = [1,2,3,4]` → `"23:41"` (các giờ hợp lệ: `12:34, 12:43, 13:24, 13:42, 14:23, 14:32, 21:34, 21:43, 23:14, 23:41`).

**Ví dụ 2:** `arr = [5,5,5,5]` → `""` (không có giờ hợp lệ vì `55:55` sai).

**Ràng buộc:** `arr.length == 4`; `0 <= arr[i] <= 9`.

In [1]:
import itertools
import random
import time
from collections import Counter
from typing import List

examples = [([1, 2, 3, 4], "23:41"), ([5, 5, 5, 5], ""), ([0, 0, 0, 0], "00:00"), ([2, 0, 6, 6], "06:26")]
for a, want in examples:
    print(f"arr = {a} -> {want!r}")

arr = [1, 2, 3, 4] -> '23:41'
arr = [5, 5, 5, 5] -> ''
arr = [0, 0, 0, 0] -> '00:00'
arr = [2, 0, 6, 6] -> '06:26'


## 3. Kiến thức nền

### 3.1 Hoán vị và đa tập chữ số

`itertools.permutations(arr)` sinh mọi cách xếp `4` chữ số (có thể lặp lại cùng bộ khi chữ số trùng). Dùng `permutations` trên **vị trí** nên các chữ số trùng nhau sinh hoán vị trùng — không ảnh hưởng đáp án (chỉ chọn cực đại).

In [2]:
perms = list(itertools.permutations([1, 2, 3, 4]))
print(len(perms), perms[:3])
print("với chữ số trùng:", len(list(itertools.permutations([1, 1, 2, 3]))), "hoán vị (6 bản trùng) -> chỉ", len(set(itertools.permutations([1, 1, 2, 3]))), "bộ phân biệt")

24 [(1, 2, 3, 4), (1, 2, 4, 3), (1, 3, 2, 4)]
với chữ số trùng: 24 hoán vị (6 bản trùng) -> chỉ 12 bộ phân biệt


### 3.2 Điều kiện hợp lệ và thứ tự "muộn nhất"

Bộ `(a, b, c, d)` tạo `HH = 10a + b`, `MM = 10c + d`. Hợp lệ khi `HH ≤ 23` và `MM ≤ 59`. So sánh "muộn" theo tổng số phút `60·HH + MM`; vì chuỗi `"HH:MM"` có độ dài cố định nên so sánh **từ điển chuỗi** cho cùng thứ tự.

In [3]:
def valid_time(a, b, c, d):
    h, m = 10 * a + b, 10 * c + d
    return h <= 23 and m <= 59


def fmt(a, b, c, d):
    return f"{a}{b}:{c}{d}"


print(valid_time(2, 3, 4, 1), valid_time(2, 4, 0, 0), valid_time(1, 2, 6, 0), fmt(0, 6, 2, 6))
print("so theo phút = so theo chuỗi:", (60 * 23 + 41 > 60 * 21 + 43) == ("23:41" > "21:43"))

True False False 06:26
so theo phút = so theo chuỗi: True


### 3.3 Đếm chữ số (đa tập)

`Counter` của bốn chữ số cho phép so sánh "có dùng đúng các chữ số của `arr`" mà không quan tâm thứ tự: `Counter(arr) == Counter(digits_of_time)`.

### 3.4 Ràng buộc hẹp cho tham lam

Chữ số hàng chục của giờ chỉ có thể là `0, 1, 2`; nếu là `2` thì chữ số hàng đơn vị `≤ 3`; chữ số hàng chục của phút `≤ 5`. Tham lam theo từng vị trí cần kiểm tra **phần còn lại còn lập được** thời gian hợp lệ.

In [4]:
print("h1 ∈ {0,1,2}; nếu h1 = 2 thì h2 ≤ 3; m1 ≤ 5; m2 bất kỳ")
print([(h1, h2) for h1 in range(3) for h2 in range(10) if 10 * h1 + h2 <= 23][-5:], "...", sum(1 for h1 in range(3) for h2 in range(10) if 10 * h1 + h2 <= 23), "giờ hợp lệ")

h1 ∈ {0,1,2}; nếu h1 = 2 thì h2 ≤ 3; m1 ≤ 5; m2 bất kỳ
[(1, 9), (2, 0), (2, 1), (2, 2), (2, 3)] ... 24 giờ hợp lệ


## 4. Ý tưởng nền tảng / trực giác

`arr = [1, 2, 3, 4]`. Muốn muộn nhất ⇒ giờ lớn nhất rồi phút lớn nhất.

- Giờ lớn nhất tạo được từ hai chữ số: `23` (dùng `2`, `3`). Phần còn lại `{1, 4}` tạo phút lớn nhất `41` (≤ 59) ⇒ `23:41`.
- Không có giờ lớn hơn `23`; vậy `23:41`.

Với `[2, 0, 6, 6]`: giờ lớn nhất `20`? Còn `{6, 6}` ⇒ phút `66 > 59` không hợp lệ. Thử giờ `06` (chữ số `0, 6`) còn `{2, 6}` ⇒ phút `62` ✗, `26` ✓ ⇒ `06:26`. Đây là ví dụ tham lam đơn giản (chọn giờ lớn nhất) **sai**: phải kiểm tra phần còn lại.

In [5]:
arr = [2, 0, 6, 6]
cands = sorted({fmt(*p) for p in itertools.permutations(arr) if valid_time(*p)})
print("mọi giờ hợp lệ:", cands, "-> lớn nhất:", cands[-1] if cands else "")

mọi giờ hợp lệ: ['06:26'] -> lớn nhất: 06:26


## 5. Cách 1 — Thử mọi hoán vị

**Ý tưởng:** `itertools.permutations(arr)` cho 24 bộ; lọc các bộ hợp lệ (`HH ≤ 23`, `MM ≤ 59`); lấy bộ cho tổng số phút lớn nhất; định dạng `"HH:MM"`; không có thì `""`.

In [6]:
def largest_time_permutations(arr):
    best = -1
    for a, b, c, d in itertools.permutations(arr):
        h, m = 10 * a + b, 10 * c + d
        if h <= 23 and m <= 59:
            best = max(best, 60 * h + m)                 # so theo tổng số phút
    if best < 0:
        return ""
    return f"{best // 60:02d}:{best % 60:02d}"           # định dạng có chữ số 0 đầu


for a, want in examples:
    assert largest_time_permutations(a) == want
print([largest_time_permutations(a) for a, _ in examples])

['23:41', '', '00:00', '06:26']


In [7]:
# Code nộp LeetCode cho Cách 1
import itertools


class SolutionPermutations:
    def largestTimeFromDigits(self, arr: List[int]) -> str:
        best = -1
        for a, b, c, d in itertools.permutations(arr):
            h, m = 10 * a + b, 10 * c + d
            if h <= 23 and m <= 59:
                best = max(best, 60 * h + m)
        return "" if best < 0 else f"{best // 60:02d}:{best % 60:02d}"

**Độ phức tạp:** `24` hoán vị `× O(1)`: `O(1)` thời gian, `O(1)` bộ nhớ.

**Nút thắt:** chỉ có `24` khả năng nên không có nút thắt; nhưng các góc nhìn khác (liệt kê giờ, tham lam) hay gặp ở bài tương tự với nhiều chữ số hơn.

## 6. Ý tưởng thuật toán / bước đột phá

Liệt kê trực tiếp là đủ. Hai biến thể: (1) duyệt `1440` thời điểm từ muộn đến sớm và dừng ở thời điểm đầu tiên có **cùng đa tập chữ số**; (2) tham lam từng vị trí với kiểm tra khả thi (tổng quát hơn khi số chữ số lớn).

## 7. Các cách tiếp theo

Ngoài Cách 1 (mục 5), ta có thêm 3 cách: **Cách 2** duyệt giờ từ muộn đến sớm, **Cách 3** tham lam từng vị trí, **Cách 4** quay lui theo chữ số giảm dần.

### Cách 2 — Duyệt `1440` thời điểm từ muộn đến sớm

**Giải thích từng đoạn:**

1. `count = Counter(arr)`.
2. Với `h` từ `23` xuống `0`, `m` từ `59` xuống `0`: nếu đa tập chữ số của `HHMM` bằng `count` thì trả về ngay `"HH:MM"` (thời điểm đầu tiên gặp là muộn nhất).
3. Không có ⇒ `""`.

In [8]:
def largest_time_scan(arr):
    need = Counter(arr)
    for h in range(23, -1, -1):
        for m in range(59, -1, -1):
            if Counter([h // 10, h % 10, m // 10, m % 10]) == need:    # cùng đa tập chữ số (kể cả chữ số trùng)
                return f"{h:02d}:{m:02d}"
    return ""


for a, want in examples:
    assert largest_time_scan(a) == want
print([largest_time_scan(a) for a, _ in examples])

['23:41', '', '00:00', '06:26']


In [9]:
# Code nộp LeetCode cho Cách 2
from collections import Counter


class SolutionScan:
    def largestTimeFromDigits(self, arr: List[int]) -> str:
        need = Counter(arr)
        for h in range(23, -1, -1):
            for m in range(59, -1, -1):
                if Counter([h // 10, h % 10, m // 10, m % 10]) == need:
                    return f"{h:02d}:{m:02d}"
        return ""

**Độ phức tạp:** `≤ 1440` lần so sánh đa tập: `O(1)`.

### Cách 3 — Tham lam từng vị trí (có kiểm tra khả thi)

**Ý tưởng:** chọn lần lượt `d1 d2 : d3 d4`. Với mỗi vị trí, thử các chữ số còn lại **giảm dần**; chọn chữ số lớn nhất mà (i) thoả ràng buộc cục bộ (`d1 ≤ 2`; `d1 = 2 ⇒ d2 ≤ 3`; `d3 ≤ 5`) và (ii) phần còn lại **vẫn lập được** thời gian hợp lệ (kiểm tra bằng thử các chữ số còn lại cho các vị trí sau). Vì chỉ `4` chữ số nên kiểm tra khả thi bằng thử nhanh.

In [10]:
def largest_time_greedy(arr):
    digits = sorted(arr, reverse=True)
    limits = [lambda d, ch: d <= 2,                         # chục giờ
              lambda d, ch: (d <= 3) if ch[0] == 2 else True,   # đơn vị giờ (nếu chục giờ = 2 thì <= 3)
              lambda d, ch: d <= 5,                         # chục phút
              lambda d, ch: True]                           # đơn vị phút

    def feasible(chosen, pool):
        """Có cách xếp pool vào các vị trí còn lại để thành thời gian hợp lệ không?"""
        if not pool:
            return True
        pos = len(chosen)
        for i, d in enumerate(pool):
            if i and pool[i - 1] == d:
                continue
            if limits[pos](d, chosen) and feasible(chosen + [d], pool[:i] + pool[i + 1:]):
                return True
        return False

    chosen, pool = [], digits[:]
    for pos in range(4):
        picked = None
        for i, d in enumerate(pool):                        # pool giảm dần: thử chữ số lớn nhất trước
            if limits[pos](d, chosen) and feasible(chosen + [d], pool[:i] + pool[i + 1:]):
                picked = i
                break
        if picked is None:
            return ""
        chosen.append(pool.pop(picked))
    return f"{chosen[0]}{chosen[1]}:{chosen[2]}{chosen[3]}"


for a, want in examples:
    assert largest_time_greedy(a) == want
print([largest_time_greedy(a) for a, _ in examples])

['23:41', '', '00:00', '06:26']


In [11]:
# Code nộp LeetCode cho Cách 3
class SolutionGreedy:
    def largestTimeFromDigits(self, arr: List[int]) -> str:
        ok = [lambda d, ch: d <= 2, lambda d, ch: d <= 3 if ch[0] == 2 else True, lambda d, ch: d <= 5, lambda d, ch: True]

        def feasible(chosen, pool):
            if not pool:
                return True
            for i, d in enumerate(pool):
                if ok[len(chosen)](d, chosen) and feasible(chosen + [d], pool[:i] + pool[i + 1:]):
                    return True
            return False

        chosen, pool = [], sorted(arr, reverse=True)
        for pos in range(4):
            for i, d in enumerate(pool):
                if ok[pos](d, chosen) and feasible(chosen + [d], pool[:i] + pool[i + 1:]):
                    chosen.append(pool.pop(i))
                    break
            else:
                return ""
        return f"{chosen[0]}{chosen[1]}:{chosen[2]}{chosen[3]}"

**Độ phức tạp:** tối đa `O(4! )` thao tác (kiểm tra khả thi đệ quy), nhưng tổng quát hoá tốt: mỗi vị trí chỉ thử chữ số lớn nhất khả thi.

### Cách 4 — Quay lui theo chữ số giảm dần (cắt nhánh)

**Ý tưởng:** quay lui điền từng vị trí theo thứ tự chữ số **giảm dần**; nhánh đầu tiên điền xong toàn bộ vị trí hợp lệ chính là thời gian lớn nhất theo thứ tự từ điển (vì ở mỗi vị trí ta thử chữ số lớn trước). Không cần so sánh nhiều kết quả: trả về ngay khi tìm được.

In [12]:
def largest_time_backtrack(arr):
    digits = sorted(arr, reverse=True)
    used = [False] * 4
    path = []

    def ok(pos, d):
        if pos == 0:
            return d <= 2
        if pos == 1:
            return d <= 3 if path[0] == 2 else True
        if pos == 2:
            return d <= 5
        return True

    def dfs(pos):
        if pos == 4:
            return True
        prev = None
        for i in range(4):
            if used[i] or digits[i] == prev:                # bỏ qua chữ số trùng vừa thử ở cùng vị trí
                continue
            if not ok(pos, digits[i]):
                continue
            prev = digits[i]
            used[i] = True
            path.append(digits[i])
            if dfs(pos + 1):
                return True
            path.pop()
            used[i] = False
        return False

    return f"{path[0]}{path[1]}:{path[2]}{path[3]}" if dfs(0) else ""


for a, want in examples:
    assert largest_time_backtrack(a) == want
print([largest_time_backtrack(a) for a, _ in examples])

['23:41', '', '00:00', '06:26']


In [13]:
# Code nộp LeetCode cho Cách 4
class SolutionBacktrack:
    def largestTimeFromDigits(self, arr: List[int]) -> str:
        digits = sorted(arr, reverse=True)
        used, path = [False] * 4, []

        def ok(pos, d):
            if pos == 0:
                return d <= 2
            if pos == 1:
                return d <= 3 if path[0] == 2 else True
            if pos == 2:
                return d <= 5
            return True

        def dfs(pos):
            if pos == 4:
                return True
            prev = None
            for i in range(4):
                if used[i] or digits[i] == prev or not ok(pos, digits[i]):
                    continue
                prev = digits[i]
                used[i] = True
                path.append(digits[i])
                if dfs(pos + 1):
                    return True
                path.pop()
                used[i] = False
            return False

        return f"{path[0]}{path[1]}:{path[2]}{path[3]}" if dfs(0) else ""

**Độ phức tạp:** tối đa `4!` nút; cắt nhánh nên thường ít hơn. `O(1)` bộ nhớ.

## 8. TRACE từng bước

**Trace Cách 1** với `arr = [2, 0, 6, 6]` (liệt kê hoán vị phân biệt, lọc hợp lệ):

| bộ `(a,b,c,d)` | `HH` | `MM` | hợp lệ | phút |
|---|---|---|---|---|
| `(2,0,6,6)` | 20 | 66 | không | — |
| `(0,2,6,6)` | 02 | 66 | không | — |
| `(0,6,2,6)` | 06 | 26 | **có** | 386 |
| `(0,6,6,2)` | 06 | 62 | không | — |
| `(2,6,…)` | 26 | … | không (`26 > 23`) | — |
| `(6,0,…)` | 60 | … | không | — |

Chỉ `06:26` hợp lệ ⇒ đáp án `"06:26"` ✓. Hàm trace liệt kê các bộ phân biệt:

In [14]:
def trace_permutations(arr):
    print(f"arr = {arr}")
    seen, best = set(), -1
    for p in sorted(set(itertools.permutations(arr)), reverse=True):
        a, b, c, d = p
        h, m = 10 * a + b, 10 * c + d
        ok = h <= 23 and m <= 59
        if ok:
            best = max(best, 60 * h + m)
        print(f"  {fmt(*p)}  HH={h:2d} MM={m:2d}  {'hợp lệ, ' + str(60 * h + m) + ' phút' if ok else 'không hợp lệ'}")
    ans = "" if best < 0 else f"{best // 60:02d}:{best % 60:02d}"
    print("=>", repr(ans))
    return ans


assert trace_permutations([2, 0, 6, 6]) == "06:26"

arr = [2, 0, 6, 6]
  66:20  HH=66 MM=20  không hợp lệ
  66:02  HH=66 MM= 2  không hợp lệ
  62:60  HH=62 MM=60  không hợp lệ
  62:06  HH=62 MM= 6  không hợp lệ
  60:62  HH=60 MM=62  không hợp lệ
  60:26  HH=60 MM=26  không hợp lệ
  26:60  HH=26 MM=60  không hợp lệ
  26:06  HH=26 MM= 6  không hợp lệ
  20:66  HH=20 MM=66  không hợp lệ
  06:62  HH= 6 MM=62  không hợp lệ
  06:26  HH= 6 MM=26  hợp lệ, 386 phút
  02:66  HH= 2 MM=66  không hợp lệ
=> '06:26'


**Trace Cách 3 (tham lam)** với `[2, 0, 6, 6]`, `digits = [6, 6, 2, 0]`:

- Vị trí 0: thử `6`: vi phạm `d ≤ 2`; thử `6`; thử `2`: thoả; kiểm tra khả thi phần còn lại `{6, 6, 0}` cho `h2 m1 m2`: `h2 ≤ 3` ⇒ chỉ `0`; còn `{6, 6}` cho `m1 m2`: `m1 ≤ 5` ✗ ⇒ không khả thi. Thử `0`: thoả; còn `{6, 6, 2}`: `h2` bất kỳ (`h1 = 0`) ⇒ chọn `6`; rồi `m1 ≤ 5`: `2` ✓, `m2 = 6` ⇒ khả thi ⇒ chọn `0`.
- Vị trí 1: `6`; vị trí 2: `2`; vị trí 3: `6` ⇒ `06:26` ✓.

## 9. Code nộp LeetCode

Chọn **Cách 1** (thử 24 hoán vị: ngắn, rõ, không sai). Giữ nguyên `class Solution`, `largestTimeFromDigits(self, arr)`:

In [15]:
class Solution:
    def largestTimeFromDigits(self, arr: List[int]) -> str:
        best = -1                                          # tổng số phút lớn nhất của một giờ hợp lệ
        for a, b, c, d in itertools.permutations(arr):     # 24 cách xếp 4 chữ số
            h, m = 10 * a + b, 10 * c + d
            if h <= 23 and m <= 59:                        # giờ hợp lệ
                best = max(best, 60 * h + m)
        return "" if best < 0 else f"{best // 60:02d}:{best % 60:02d}"      # :02d giữ chữ số 0 đầu


print([Solution().largestTimeFromDigits(list(a)) for a, _ in examples])      # ['23:41', '', '00:00', '06:26']
assert all(Solution().largestTimeFromDigits(list(a)) == want for a, want in examples)

['23:41', '', '00:00', '06:26']


## 10. Vì sao đúng

**(a) Mọi lời giải là một hoán vị.** Thời gian `"HH:MM"` dùng đúng bốn chữ số của `arr`, mỗi chữ số một lần ⇔ chuỗi `H1 H2 M1 M2` là một hoán vị (theo vị trí) của `arr`. `permutations(arr)` sinh mọi hoán vị theo vị trí (kể cả bản trùng khi chữ số trùng), nên tập các bộ `(a, b, c, d)` hợp lệ được duyệt đầy đủ.

**(b) Hợp lệ ⇔ `HH ≤ 23` và `MM ≤ 59`.** Đó là định nghĩa của thời gian 24 giờ (`00:00` tới `23:59`).

**(c) Thứ tự muộn nhất.** Thời gian ứng với tổng số phút `T = 60·HH + MM ∈ [0, 1439]`, ánh xạ **đơn ánh** từ thời gian hợp lệ sang `T`, tăng theo thời điểm; do đó muộn nhất ⇔ `T` lớn nhất. Định dạng ngược: `HH = T // 60`, `MM = T % 60` với `:02d`.

**(d) Cách 2.** Duyệt `(h, m)` giảm dần theo `(h, m)` từ điển tương đương giảm dần theo `T`; thời điểm đầu tiên có đa tập chữ số của `HHMM` bằng `Counter(arr)` chính là muộn nhất. Đa tập chữ số bằng nhau ⇔ `HHMM` là hoán vị của `arr`.

**(e) Cách 3, 4 (tham lam / quay lui theo chữ số giảm dần).** Thứ tự từ điển của chuỗi `"HH:MM"` trùng với thứ tự thời gian (độ dài cố định, dấu `:` ở vị trí cố định). Chọn chữ số lớn nhất ở vị trí đầu mà **vẫn tồn tại** cách hoàn thiện hợp lệ cho các vị trí sau, rồi lặp lại: đây là cách dựng chuỗi lớn nhất theo từ điển (kiểm tra khả thi bắt buộc — xem ví dụ `[2,0,6,6]`). Quay lui thử chữ số lớn trước nên nhánh thành công đầu tiên là lớn nhất.

**(f) Chữ số trùng.** Ở Cách 4, bỏ qua chữ số bằng chữ số vừa thử ở cùng vị trí (`prev`) tránh lặp nhánh; không ảnh hưởng tính đúng vì hai chữ số bằng nhau sinh cùng nhánh.

Ta kiểm chứng bằng máy: mọi cách khớp oracle **duyệt mọi bộ `abcd` từ `0000` tới `9999`** trên **mọi** mảng `arr` (`10^4` mảng) ở mục 12D.

## 11. Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử 24 hoán vị | `O(1)` | `O(1)` | **đáp án chính** |
| 2. Duyệt 1440 thời điểm | `O(1)` (≤ 1440 so sánh) | `O(1)` | trực quan |
| 3. Tham lam + khả thi | `O(1)` | `O(1)` | tổng quát hoá cho nhiều chữ số |
| 4. Quay lui chữ số giảm dần | `O(1)` (≤ 24 nút) | `O(1)` | dừng sớm |

Đo thực tế (trung bình trên mọi mảng `4` chữ số):

In [16]:
def bench(fn, arrs, repeat=3):
    best = 1e9
    for _ in range(repeat):
        t0 = time.perf_counter()
        for a in arrs:
            fn(a)
        best = min(best, (time.perf_counter() - t0) / len(arrs))
    return best


rng = random.Random(949)
arrs = [[rng.randint(0, 9) for _ in range(4)] for _ in range(1000)]
for name, fn in [("permutations", largest_time_permutations), ("scan", largest_time_scan), ("greedy", largest_time_greedy), ("backtrack", largest_time_backtrack)]:
    print(f"{name:13s}: {bench(fn, arrs) * 1e6:8.2f} us / lần gọi")

permutations :     4.15 us / lần gọi


scan         :  1165.97 us / lần gọi
greedy       :     4.91 us / lần gọi
backtrack    :     4.83 us / lần gọi


## 12. Bộ test

Chia 6 nhóm: (A) ví dụ đề, (B) edge case, (C) phản ví dụ, (D) vét cạn **mọi** mảng `4` chữ số (`10^4`) với oracle duyệt `abcd`, (E) tính chất, (F) giới hạn (đã vét cạn ở D).

### A. Ví dụ của đề

In [17]:
def oracle(arr):
    """Duyệt MỌI bộ 4 chữ số abcd (0000..9999): hợp lệ nếu cùng đa tập chữ số với arr và là giờ hợp lệ; lấy muộn nhất theo chuỗi HH:MM."""
    target = sorted(arr)
    best = None
    for n in range(10000):
        digs = [n // 1000, n // 100 % 10, n // 10 % 10, n % 10]
        if sorted(digs) != target:
            continue
        a, b, c, d = digs
        if 10 * a + b <= 23 and 10 * c + d <= 59:
            s = f"{a}{b}:{c}{d}"
            if best is None or s > best:
                best = s
    return best or ""


IMPLS = {
    "permutations": largest_time_permutations,
    "scan": largest_time_scan,
    "greedy": largest_time_greedy,
    "backtrack": largest_time_backtrack,
    "Solution": lambda a: Solution().largestTimeFromDigits(list(a)),
    "SolutionPermutations": lambda a: SolutionPermutations().largestTimeFromDigits(list(a)),
    "SolutionScan": lambda a: SolutionScan().largestTimeFromDigits(list(a)),
    "SolutionGreedy": lambda a: SolutionGreedy().largestTimeFromDigits(list(a)),
    "SolutionBacktrack": lambda a: SolutionBacktrack().largestTimeFromDigits(list(a)),
}


def check(arr, want=None, names=None):
    if want is None:
        want = oracle(arr)
    for name in (names or IMPLS):
        got = IMPLS[name](list(arr))
        assert got == want, (name, arr, got, want)


for a, want in examples:
    assert oracle(a) == want
    check(a, want)
print("ví dụ đề: oracle duyệt abcd và", len(IMPLS), "cài đặt đúng")

ví dụ đề: oracle duyệt abcd và 9 cài đặt đúng


### B. Edge case

- Toàn `0`: `00:00`.
- Toàn `9`/toàn `5`: không hợp lệ (`""`).
- Có đúng một giờ hợp lệ (như `[2,0,6,6]`).
- Chữ số trùng nhiều (`[1,1,1,1]` → `11:11`).
- `[2,3,5,9]` → `23:59`.
- `[2,4,6,0]` → `20:46`; `[1,9,6,0]` → `19:06`.
- `[0,0,6,6]` → `06:06`; `[6,6,6,0]` → `""` (ba chữ số `6` không thể xếp thành giờ lẻ phút hợp lệ).
- Giờ `24` bị loại: `[2,4,0,0]` → `20:40` (không phải `24:00`).
- `[0,6,2,9]` → `09:26`.

In [18]:
cases = [
    ([0, 0, 0, 0], "00:00"), ([9, 9, 9, 9], ""), ([5, 5, 5, 5], ""), ([1, 1, 1, 1], "11:11"), ([2, 3, 5, 9], "23:59"),
    ([2, 4, 6, 0], "20:46"), ([1, 9, 6, 0], "19:06"), ([0, 0, 6, 6], "06:06"), ([2, 0, 6, 6], "06:26"), ([6, 6, 6, 0], ""),
    ([2, 4, 0, 0], "20:40"), ([0, 6, 2, 9], "09:26"),
]
for a, want in cases:
    check(a, want)
print(len(cases), "edge case: oracle và mọi cài đặt đúng")

12 edge case: oracle và mọi cài đặt đúng


### C. Phản ví dụ: cài đặt sai phải bị bắt

In [19]:
def bug_hour_up_to_29(arr):                            # chấp nhận giờ 24..29
    best = -1
    for a, b, c, d in itertools.permutations(arr):
        h, m = 10 * a + b, 10 * c + d
        if h <= 29 and m <= 59:
            best = max(best, 60 * h + m)
    return "" if best < 0 else f"{best // 60:02d}:{best % 60:02d}"


def bug_minute_up_to_69(arr):                          # chấp nhận phút 60..69
    best = -1
    for a, b, c, d in itertools.permutations(arr):
        h, m = 10 * a + b, 10 * c + d
        if h <= 23 and m <= 69:
            best = max(best, 60 * h + m)
    return "" if best < 0 else f"{best // 60:02d}:{best % 60:02d}"


def bug_greedy_no_feasibility(arr):                    # chọn giờ lớn nhất bằng tham lam, không kiểm tra phần còn lại
    digits = sorted(arr, reverse=True)
    h1 = next((d for d in digits if d <= 2), None)
    if h1 is None:
        return ""
    rest = digits[:]
    rest.remove(h1)
    h2 = next((d for d in rest if d <= (3 if h1 == 2 else 9)), None)
    if h2 is None:
        return ""
    rest.remove(h2)
    m1 = next((d for d in rest if d <= 5), None)
    if m1 is None:
        return ""
    rest.remove(m1)
    return f"{h1}{h2}:{m1}{rest[0]}"


def bug_reuse_digit(arr):                              # cho phép dùng một chữ số nhiều lần
    best = -1
    for a, b, c, d in itertools.product(set(arr), repeat=4):
        h, m = 10 * a + b, 10 * c + d
        if h <= 23 and m <= 59:
            best = max(best, 60 * h + m)
    return "" if best < 0 else f"{best // 60:02d}:{best % 60:02d}"


def bug_no_zero_pad(arr):                              # không đệm chữ số 0
    best = -1
    for a, b, c, d in itertools.permutations(arr):
        h, m = 10 * a + b, 10 * c + d
        if h <= 23 and m <= 59:
            best = max(best, 60 * h + m)
    return "" if best < 0 else f"{best // 60}:{best % 60}"


def bug_none(arr):                                     # trả None thay vì ""
    r = largest_time_permutations(arr)
    return r or None


def bug_earliest(arr):                                 # trả thời gian SỚM nhất
    best = 10 ** 9
    for a, b, c, d in itertools.permutations(arr):
        h, m = 10 * a + b, 10 * c + d
        if h <= 23 and m <= 59:
            best = min(best, 60 * h + m)
    return "" if best == 10 ** 9 else f"{best // 60:02d}:{best % 60:02d}"


BUGS = [bug_hour_up_to_29, bug_minute_up_to_69, bug_greedy_no_feasibility, bug_reuse_digit, bug_no_zero_pad, bug_none, bug_earliest]
all_arrays = list(itertools.product(range(10), repeat=4))
for bug in BUGS:
    bad = next((list(a) for a in all_arrays if bug(list(a)) != oracle(list(a))), None)
    assert bad is not None, bug.__name__
    print(f"{bug.__name__:26s} sai trên {bad}: ra {bug(bad)!r}, đúng {oracle(bad)!r}")

bug_hour_up_to_29          sai trên [0, 0, 2, 4]: ra '24:00', đúng '20:40'


bug_minute_up_to_69        sai trên [0, 0, 2, 6]: ra '21:00', đúng '20:06'


bug_greedy_no_feasibility  sai trên [0, 2, 6, 6]: ra '', đúng '06:26'
bug_reuse_digit            sai trên [0, 0, 0, 1]: ra '11:11', đúng '10:00'
bug_no_zero_pad            sai trên [0, 0, 0, 0]: ra '0:0', đúng '00:00'


bug_none                   sai trên [0, 6, 6, 6]: ra None, đúng ''
bug_earliest               sai trên [0, 0, 0, 1]: ra '00:01', đúng '10:00'


### D. Vét cạn **mọi** mảng `4` chữ số

Mọi `10^4 = 10 000` mảng `arr` (mọi tổ hợp chữ số `0..9` kể cả thứ tự — nhiều mảng là hoán vị của nhau; số đa tập khác nhau là `715`). Oracle: duyệt `abcd` từ `0000` tới `9999`.

In [20]:
t0 = time.perf_counter()
cnt = 0
oracle_cache = {}
nonempty = 0
for arr in all_arrays:
    key = tuple(sorted(arr))
    if key not in oracle_cache:
        oracle_cache[key] = oracle(list(arr))
    want = oracle_cache[key]
    nonempty += want != ""
    for name in IMPLS:
        assert IMPLS[name](list(arr)) == want, (name, arr)
    cnt += 1
print(cnt, "mảng (715 đa tập khác nhau):", len(IMPLS), "cài đặt khớp oracle;", nonempty, "mảng có giờ hợp lệ;", round(time.perf_counter() - t0, 1), "s")
assert cnt == 10000 and len(oracle_cache) == 715

10000 mảng (715 đa tập khác nhau): 9 cài đặt khớp oracle; 5935 mảng có giờ hợp lệ; 27.6 s


### E. Tính chất

(1) Kết quả không phụ thuộc **thứ tự** của `arr`; (2) kết quả không rỗng ⇔ tồn tại giờ hợp lệ; (3) nếu `HH:MM` là kết quả thì mọi chữ số của nó đúng bằng đa tập của `arr`; (4) số mảng cho kết quả rỗng khớp đếm trực tiếp.

In [21]:
rng = random.Random(20240949)
for _ in range(3000):
    arr = [rng.randint(0, 9) for _ in range(4)]
    want = oracle_cache[tuple(sorted(arr))]
    sh = arr[:]
    rng.shuffle(sh)
    assert Solution().largestTimeFromDigits(sh) == want                                 # (1)
    assert (want != "") == any(valid_time(*p) for p in itertools.permutations(arr))     # (2)
    if want:
        assert sorted(int(ch) for ch in want.replace(":", "")) == sorted(arr)           # (3)
        h, m = int(want[:2]), int(want[3:])
        assert 0 <= h <= 23 and 0 <= m <= 59
empty_count = sum(1 for k, v in oracle_cache.items() if v == "")
direct_empty = sum(1 for k in oracle_cache if not any(valid_time(*p) for p in itertools.permutations(k)))
assert empty_count == direct_empty                                                         # (4)
print("3000 mảng ngẫu nhiên: 3 tính chất đúng; số đa tập không có giờ hợp lệ:", empty_count, "/ 715")

3000 mảng ngẫu nhiên: 3 tính chất đúng; số đa tập không có giờ hợp lệ: 326 / 715


### F. Giới hạn

Không gian đầu vào nhỏ (`10^4`), đã vét cạn toàn bộ ở D. Các trường hợp cực đoan: chữ số lớn nhất (`9999`, `5959`...), chữ số giống nhau, chữ số `0` nhiều.

In [22]:
extremes = [[9, 9, 9, 9], [5, 9, 5, 9], [2, 3, 5, 9], [0, 0, 0, 0], [0, 0, 0, 1], [0, 0, 2, 3], [2, 2, 2, 2], [2, 3, 2, 3], [1, 2, 6, 7], [6, 0, 0, 0]]
for a in extremes:
    check(a, oracle_cache[tuple(sorted(a))])
print("cực đoan:", {tuple(a): Solution().largestTimeFromDigits(a) for a in extremes})

cực đoan: {(9, 9, 9, 9): '', (5, 9, 5, 9): '', (2, 3, 5, 9): '23:59', (0, 0, 0, 0): '00:00', (0, 0, 0, 1): '10:00', (0, 0, 2, 3): '23:00', (2, 2, 2, 2): '22:22', (2, 3, 2, 3): '23:32', (1, 2, 6, 7): '17:26', (6, 0, 0, 0): '06:00'}


## 13. Các lỗi người mới hay gặp

### Lỗi 1: chấp nhận giờ `24..29`

In [23]:
print("[2,4,0,0]:", repr(bug_hour_up_to_29([2, 4, 0, 0])), "| đúng:", repr(Solution().largestTimeFromDigits([2, 4, 0, 0])))
assert bug_hour_up_to_29([2, 4, 0, 0]) == "24:00" and Solution().largestTimeFromDigits([2, 4, 0, 0]) == "20:40"

[2,4,0,0]: '24:00' | đúng: '20:40'


**Vì sao sai:** `24:00` không phải thời gian hợp lệ trong định dạng đề (tối đa `23`). **Đúng:** `HH ≤ 23`.

### Lỗi 2: chấp nhận phút `60..69`

In [24]:
print("[1,2,6,0]:", repr(bug_minute_up_to_69([1, 2, 6, 0])), "| đúng:", repr(Solution().largestTimeFromDigits([1, 2, 6, 0])))
assert bug_minute_up_to_69([1, 2, 6, 0]) != Solution().largestTimeFromDigits([1, 2, 6, 0])

[1,2,6,0]: '22:00' | đúng: '21:06'


**Vì sao sai:** phút chỉ tới `59`. **Đúng:** `MM ≤ 59`.

### Lỗi 3: tham lam không kiểm tra phần còn lại

In [25]:
print("[2,0,6,6]:", end=" ")
try:
    print(repr(bug_greedy_no_feasibility([2, 0, 6, 6])), end=" ")
except IndexError as e:
    print("IndexError", end=" ")
print("| đúng:", repr(Solution().largestTimeFromDigits([2, 0, 6, 6])))
bad = next(list(a) for a in all_arrays if bug_greedy_no_feasibility(list(a)) != oracle(list(a)))
print("phản ví dụ nhỏ nhất:", bad, "->", repr(bug_greedy_no_feasibility(bad)), "| đúng:", repr(oracle(bad)))

[2,0,6,6]: '' | đúng: '06:26'


phản ví dụ nhỏ nhất: [0, 2, 6, 6] -> '' | đúng: '06:26'


**Vì sao sai:** chọn giờ lớn nhất trước có thể làm hai chữ số còn lại không tạo được phút hợp lệ (ví dụ `20` rồi `{6, 6}`). **Đúng:** thử mọi hoán vị, hoặc tham lam **có kiểm tra khả thi**.

### Lỗi 4: dùng một chữ số nhiều lần

In [26]:
print("[1,2,3,4]:", repr(bug_reuse_digit([1, 2, 3, 4])), "| đúng:", repr(Solution().largestTimeFromDigits([1, 2, 3, 4])))
assert bug_reuse_digit([1, 2, 3, 4]) == "23:44" and Solution().largestTimeFromDigits([1, 2, 3, 4]) == "23:41"

[1,2,3,4]: '23:44' | đúng: '23:41'


**Vì sao sai:** "mỗi chữ số đúng **một lần**" (phép hoán vị, không phải tích Descartes). **Đúng:** `permutations`.

### Lỗi 5: quên đệm số `0`

In [27]:
print("[2,0,6,6]:", repr(bug_no_zero_pad([2, 0, 6, 6])), "| đúng:", repr(Solution().largestTimeFromDigits([2, 0, 6, 6])))
assert bug_no_zero_pad([2, 0, 6, 6]) == "6:26" and Solution().largestTimeFromDigits([2, 0, 6, 6]) == "06:26"

[2,0,6,6]: '6:26' | đúng: '06:26'


**Vì sao sai:** định dạng cố định `"HH:MM"`. **Đúng:** `f"{h:02d}:{m:02d}"`.

### Lỗi 6 (bonus): trả `None` hoặc thời gian sớm nhất

In [28]:
print("[5,5,5,5]:", repr(bug_none([5, 5, 5, 5])), "| đúng:", repr(Solution().largestTimeFromDigits([5, 5, 5, 5])))
print("sớm nhất [1,2,3,4]:", repr(bug_earliest([1, 2, 3, 4])), "| đúng muộn nhất:", repr(Solution().largestTimeFromDigits([1, 2, 3, 4])))
assert bug_none([5, 5, 5, 5]) is None and bug_earliest([1, 2, 3, 4]) == "12:34"

[5,5,5,5]: None | đúng: ''
sớm nhất [1,2,3,4]: '12:34' | đúng muộn nhất: '23:41'


**Vì sao sai:** đề yêu cầu chuỗi rỗng `""` và thời gian **muộn** nhất.

## 14. Mẹo & nhận dạng pattern

**Nhận dạng:** "tạo giá trị lớn nhất/nhỏ nhất từ vài chữ số (≤ 6) với ràng buộc định dạng" → **liệt kê hoán vị** (ít khả năng), hoặc tham lam có kiểm tra khả thi cho nhiều chữ số hơn.

**Mẹo:**

1. Quy thời gian về **tổng số phút** để so sánh.
2. `:02d` đệm số `0`.
3. Ràng buộc: `HH ≤ 23`, `MM ≤ 59`.
4. Chữ số trùng: `permutations` sinh bản trùng, không ảnh hưởng cực đại.
5. Tham lam cần **kiểm tra phần còn lại**.
6. Oracle: duyệt toàn bộ không gian đầu ra/đầu vào nhỏ.

**Họ bài cùng pattern:** Next Closest Time (681, Premium), Largest Number (179), Number of Valid Clock Times (2437), Maximum Time (1736), Latest Time by Replacing Hidden Digits (1736), Largest Time ... biến thể 12 giờ.

## 15. Biến thể

1. **Giờ 12h có `AM/PM`.**
2. **Thời gian sớm nhất** (hoặc gần nhất sau một thời điểm cho trước).
3. **Cho `n = 6` chữ số** (định dạng `HH:MM:SS`).
4. **Có ký tự `?` cần điền** (Maximum Time 1736).
5. **Đếm số giờ hợp lệ** tạo được.

Cài biến thể 2 (sớm nhất), 3 (6 chữ số, giây) và 5 (đếm) và đối chiếu vét cạn:

In [29]:
def earliest_time(arr):
    best = None
    for a, b, c, d in itertools.permutations(arr):
        if 10 * a + b <= 23 and 10 * c + d <= 59:
            s = f"{a}{b}:{c}{d}"
            if best is None or s < best:
                best = s
    return best or ""


def largest_time_hms(arr):
    """Biến thể 3: 6 chữ số, HH:MM:SS; giờ <= 23, phút <= 59, giây <= 59."""
    best = -1
    for p in set(itertools.permutations(arr)):
        h, m, s = 10 * p[0] + p[1], 10 * p[2] + p[3], 10 * p[4] + p[5]
        if h <= 23 and m <= 59 and s <= 59:
            best = max(best, h * 3600 + m * 60 + s)
    return "" if best < 0 else f"{best // 3600:02d}:{best % 3600 // 60:02d}:{best % 60:02d}"


def count_valid_times(arr):
    return len({p for p in itertools.permutations(arr) if 10 * p[0] + p[1] <= 23 and 10 * p[2] + p[3] <= 59})


def brute_hms(arr):
    best = None
    for h in range(24):
        for m in range(60):
            for s in range(60):
                if sorted([h // 10, h % 10, m // 10, m % 10, s // 10, s % 10]) == sorted(arr):
                    t = f"{h:02d}:{m:02d}:{s:02d}"
                    if best is None or t > best:
                        best = t
    return best or ""


rng = random.Random(15)
for _ in range(300):
    arr = [rng.randint(0, 9) for _ in range(4)]
    cands = [f"{h:02d}:{m:02d}" for h in range(24) for m in range(60) if sorted([h // 10, h % 10, m // 10, m % 10]) == sorted(arr)]
    assert earliest_time(arr) == (min(cands) if cands else "")
    assert count_valid_times(arr) == len(cands)
for _ in range(60):
    arr6 = [rng.randint(0, 9) for _ in range(6)]
    assert largest_time_hms(arr6) == brute_hms(arr6)
print("sớm nhất, đếm số giờ hợp lệ, HH:MM:SS (6 chữ số): khớp vét cạn")

sớm nhất, đếm số giờ hợp lệ, HH:MM:SS (6 chữ số): khớp vét cạn


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử 24 hoán vị | lọc hợp lệ, lấy phút lớn nhất | `O(1)` | `O(1)` | **đáp án chính** |
| 2. Duyệt 1440 thời điểm | từ muộn đến sớm, so đa tập | `O(1)` | `O(1)` | trực quan |
| 3. Tham lam + khả thi | chữ số lớn nhất còn khả thi | `O(1)` | `O(1)` | tổng quát hoá |
| 4. Quay lui giảm dần | dừng ở nhánh đầu | `O(1)` | `O(1)` | cắt nhánh |

Cả bốn khớp oracle duyệt `abcd` trên **mọi** `10^4` mảng đầu vào (mục 12).

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test cuối cell.

### Bài 1 (đoán output)

Đoán trước rồi chạy:

In [30]:
print(repr(Solution().largestTimeFromDigits([1, 1, 5, 9])), repr(Solution().largestTimeFromDigits([2, 5, 5, 6])), repr(Solution().largestTimeFromDigits([0, 7, 6, 0])), repr(Solution().largestTimeFromDigits([8, 8, 8, 8])))

'19:51' '' '07:06' ''


### Bài 2 (tính tay)

`arr = [0, 2, 5, 9]`. (a) Liệt kê các giờ hợp lệ. (b) Muộn nhất? (c) Sớm nhất? Kiểm tra ở mục 19.

### Bài 3 (điền TODO)

Hoàn thiện thử hoán vị:

In [31]:
def permutations_todo(arr):
    best = -1
    for a, b, c, d in itertools.permutations(arr):
        pass            # TODO 1: h, m; nếu h <= 23 và m <= 59 thì cập nhật best bằng 60*h + m
    return ""           # TODO 2: định dạng f"{best // 60:02d}:{best % 60:02d}" (hoặc "" nếu best < 0)


# Khi làm xong, bỏ comment để kiểm tra:
# assert [permutations_todo(a) for a, _ in examples] == [w for _, w in examples]

### Bài 4 (tự viết lại)

Không nhìn lời giải, viết `class MySolution` bằng **duyệt `1440` thời điểm từ muộn đến sớm**:

In [32]:
class MySolution:
    def largestTimeFromDigits(self, arr):
        pass            # TODO


# Khi làm xong:
# assert all(MySolution().largestTimeFromDigits(list(a)) == w for a, w in examples)

### Bài 5 (mini challenge): Latest Time by Replacing Hidden Digits (1736)

Cho chuỗi `time` dạng `"hh:mm"` trong đó một số chữ số là `'?'`. Thay mọi `?` để được thời gian hợp lệ **muộn nhất**. Viết `maximum_time(time)` và kiểm tra bằng vét cạn `1440` thời điểm.

In [33]:
def maximum_time(time):
    pass            # TODO


# Khi làm xong:
# assert maximum_time("2?:?0") == "23:50" and maximum_time("0?:3?") == "09:39" and maximum_time("1?:22") == "19:22"

## 18. Gợi ý 3 mức

**Bài 1**

- Mức 1: thử hoán vị, lấy phút lớn nhất.
- Mức 2: `[1,1,5,9]` có giờ hợp lệ `11:59, 15:19, 19:15, 19:51`; `[2,5,5,6]` và `[8,8,8,8]` không có giờ nào (chục giờ chỉ có thể là `0/1/2`, đơn vị giờ sau `2` phải `≤ 3`).
- Mức 3: kết quả là `'19:51'`, `''`, `'07:06'` (giờ hợp lệ `06:07`, `07:06`), `''`.

**Bài 2**

- Mức 1: chữ số `0,2,5,9`.
- Mức 2: chục giờ `0` hoặc `2`; chục phút `≤ 5`.
- Mức 3: giờ hợp lệ là `02:59, 05:29, 09:25, 09:52, 20:59`; muộn nhất `20:59`, sớm nhất `02:59`.

**Bài 3**

- Mức 1: `h, m = 10 * a + b, 10 * c + d`.
- Mức 2: `best = max(best, 60 * h + m)`.
- Mức 3: `"" if best < 0 else f"{best // 60:02d}:{best % 60:02d}"`.

**Bài 4**

- Mức 1: hai vòng `for h in range(23, -1, -1)`, `for m in range(59, -1, -1)`.
- Mức 2: so `Counter`.
- Mức 3: trả về ngay thời điểm đầu tiên khớp.

**Bài 5**

- Mức 1: xử lý từng vị trí.
- Mức 2: `h1`: `2` nếu `h2` là `?` hoặc `≤ 3`, ngược lại `1`; `h2`: `3` nếu `h1 = 2` ngược lại `9`.
- Mức 3: `m1 = 5`, `m2 = 9`.

## 19. Đáp án tham khảo

Mỗi đáp án đều có `assert`.

### Bài 1

In [34]:
vals = tuple(Solution().largestTimeFromDigits(a) for a in ([1, 1, 5, 9], [2, 5, 5, 6], [0, 7, 6, 0], [8, 8, 8, 8]))
assert vals == tuple(oracle(a) for a in ([1, 1, 5, 9], [2, 5, 5, 6], [0, 7, 6, 0], [8, 8, 8, 8]))
print("Bài 1 OK:", vals)

Bài 1 OK: ('19:51', '', '07:06', '')


### Bài 2

Giờ hợp lệ từ `{0, 2, 5, 9}` (xem hàm), muộn nhất và sớm nhất:

In [35]:
arr = [0, 2, 5, 9]
cands = sorted({fmt(*p) for p in itertools.permutations(arr) if valid_time(*p)})
print(cands)
assert cands[-1] == Solution().largestTimeFromDigits(arr) == oracle(arr) and cands[0] == earliest_time(arr)
print("Bài 2 OK: muộn nhất", cands[-1], "| sớm nhất", cands[0])

['02:59', '05:29', '09:25', '09:52', '20:59']
Bài 2 OK: muộn nhất 20:59 | sớm nhất 02:59


### Bài 3

In [36]:
def permutations_answer(arr):
    best = -1
    for a, b, c, d in itertools.permutations(arr):
        h, m = 10 * a + b, 10 * c + d
        if h <= 23 and m <= 59:
            best = max(best, 60 * h + m)
    return "" if best < 0 else f"{best // 60:02d}:{best % 60:02d}"


assert [permutations_answer(a) for a, _ in examples] == [w for _, w in examples]
for arr in all_arrays[::7]:
    assert permutations_answer(list(arr)) == oracle_cache[tuple(sorted(arr))]
print("Bài 3 OK")

Bài 3 OK


### Bài 4

In [37]:
class MySolutionAnswer:
    def largestTimeFromDigits(self, arr):
        need = Counter(arr)
        for h in range(23, -1, -1):
            for m in range(59, -1, -1):
                if Counter([h // 10, h % 10, m // 10, m % 10]) == need:
                    return f"{h:02d}:{m:02d}"
        return ""


assert all(MySolutionAnswer().largestTimeFromDigits(list(a)) == w for a, w in examples)
for arr in all_arrays[::7]:
    assert MySolutionAnswer().largestTimeFromDigits(list(arr)) == oracle_cache[tuple(sorted(arr))]
print("Bài 4 OK")

Bài 4 OK


### Bài 5

In [38]:
def maximum_time_answer(time):
    t = list(time)
    if t[0] == "?":
        t[0] = "2" if t[1] in "?0123" else "1"
    if t[1] == "?":
        t[1] = "3" if t[0] == "2" else "9"
    if t[3] == "?":
        t[3] = "5"
    if t[4] == "?":
        t[4] = "9"
    return "".join(t)


def maximum_time_brute(time):
    best = None
    for h in range(24):
        for m in range(60):
            s = f"{h:02d}:{m:02d}"
            if all(p == "?" or p == q for p, q in zip(time, s)) and (best is None or s > best):
                best = s
    return best


assert maximum_time_answer("2?:?0") == "23:50" and maximum_time_answer("0?:3?") == "09:39" and maximum_time_answer("1?:22") == "19:22"
rng = random.Random(5)
for _ in range(500):
    s = f"{rng.randint(0, 23):02d}:{rng.randint(0, 59):02d}"
    q = "".join("?" if (ch != ":" and rng.random() < 0.4) else ch for ch in s)
    assert maximum_time_answer(q) == maximum_time_brute(q), q
print("Bài 5 OK: khớp vét cạn")

Bài 5 OK: khớp vét cạn


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (ít chữ số, ràng buộc định dạng → liệt kê hoán vị, chọn cực đại):**

```python
class Solution:
    def largestTimeFromDigits(self, arr: List[int]) -> str:
        best = -1
        for a, b, c, d in itertools.permutations(arr):
            h, m = 10 * a + b, 10 * c + d
            if h <= 23 and m <= 59:
                best = max(best, 60 * h + m)
        return "" if best < 0 else f"{best // 60:02d}:{best % 60:02d}"
```

**Checklist trước khi submit:**

- [ ] `HH ≤ 23` và `MM ≤ 59` (không `24..29`, không `60..69`)?
- [ ] Mỗi chữ số dùng **đúng một lần** (hoán vị)?
- [ ] So sánh theo **tổng số phút** (hoặc chuỗi `"HH:MM"` có độ dài cố định)?
- [ ] Định dạng `:02d`, trả `""` (không `None`) khi không có?
- [ ] Nếu dùng tham lam: có kiểm tra phần còn lại còn khả thi?
- [ ] Đã thử `[1,2,3,4] → "23:41"`, `[5,5,5,5] → ""`, `[0,0,0,0]`, `[2,0,6,6] → "06:26"`?
- [ ] Đúng tên `class Solution`, `largestTimeFromDigits(self, arr)`, không in debug.